# BITCOIN4Traders — Autonomous Hyperparameter Search (Colab)

**Goal:** Autonomously improve the PPO adversarial trading agent by searching over
YAML hyperparameters and keeping only changes that raise the Sharpe Ratio.

## Before you start

1. **Zip your project locally** (run once in your terminal):
   ```bash
   cd /home/hp17/Tradingbot
   zip -r BITCOIN4Traders.zip BITCOIN4Traders/ -x '*.pyc' -x '__pycache__/*' -x '*.parquet' -x '*.pkl'
   ```
2. Upload `BITCOIN4Traders.zip` to **Google Drive root** (My Drive).
3. Open this notebook in **Google Colab** with a **T4 GPU** runtime.
4. Run cells top-to-bottom. The loop in Step 8 runs autonomously.

---
**Architecture:**
- Steps 1–5: Setup (GPU check, Drive mount, extract code, install deps, download data)
- Step 6: Baseline training run (50 iterations)
- Step 7: Extract baseline Sharpe
- Step 8: Autonomous loop — patch YAML → train → compare → keep/revert
- Step 9: Champion retrain (full 500 iterations)
- Step 10: View results table
- Step 11: Save to Drive

## Step 1 — Check GPU

In [ ]:
!nvidia-smi
import torch
print(f"\nPyTorch CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM: {mem:.1f} GB")
else:
    print("WARNING: No GPU detected. Training will be very slow on CPU.")
    print("Go to Runtime → Change runtime type → GPU (T4)")

## Step 2 — Mount Google Drive & Extract Code

In [ ]:
import os
import zipfile
from pathlib import Path

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Locate zip
ZIP_NAME = 'BITCOIN4Traders.zip'
DRIVE_ZIP = f'/content/drive/MyDrive/{ZIP_NAME}'
EXTRACT_DIR = Path('/content/BITCOIN4Traders')

assert Path(DRIVE_ZIP).exists(), (
    f"ERROR: {DRIVE_ZIP} not found.\n"
    f"Please upload {ZIP_NAME} to your Google Drive root first."
)

# Extract (skip if already done)
if not EXTRACT_DIR.exists():
    print(f"Extracting {ZIP_NAME}...")
    with zipfile.ZipFile(DRIVE_ZIP, 'r') as zf:
        zf.extractall('/content')
    print("Extraction complete.")
else:
    print(f"{EXTRACT_DIR} already exists — skipping extraction.")

# Set working directory
os.chdir(EXTRACT_DIR)
print(f"Working directory: {os.getcwd()}")
print("Contents:", sorted(os.listdir('.')))

## Step 3 — Install Dependencies

In [ ]:
import subprocess, sys

req_path = Path('requirements.txt')
assert req_path.exists(), "requirements.txt not found in project root"

print("Installing dependencies (this may take 2–3 minutes)...")
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt', '-q'],
    capture_output=True, text=True
)
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])
    raise RuntimeError("pip install failed")
print("Dependencies installed successfully.")

# Ensure ccxt is available for data download
try:
    import ccxt
    print(f"ccxt version: {ccxt.__version__}")
except ImportError:
    print("ccxt not in requirements.txt — installing separately...")
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'ccxt', '-q'], check=True)
    print("ccxt installed.")

## Step 4 — Download BTC/USDT Historical Data (2 years, 1h candles)

In [ ]:
import ccxt
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from pathlib import Path
import time

CACHE_DIR = Path('data/cache')
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_FILE = CACHE_DIR / 'BTCUSDT_1h_colab.parquet'

if CACHE_FILE.exists():
    print(f"Loading cached data from {CACHE_FILE}")
    df = pd.read_parquet(CACHE_FILE)
    print(f"Loaded {len(df)} candles: {df.index[0]} → {df.index[-1]}")
else:
    print("Downloading BTC/USDT 1h data from Binance (2 years)...")
    exchange = ccxt.binance({'enableRateLimit': True})

    end_dt = datetime.utcnow()
    start_dt = end_dt - timedelta(days=730)  # 2 years
    since_ms = int(start_dt.timestamp() * 1000)

    all_ohlcv = []
    limit = 1000
    current_since = since_ms

    while True:
        ohlcv = exchange.fetch_ohlcv('BTC/USDT', '1h', since=current_since, limit=limit)
        if not ohlcv:
            break
        all_ohlcv.extend(ohlcv)
        last_ts = ohlcv[-1][0]
        if last_ts >= int(end_dt.timestamp() * 1000):
            break
        current_since = last_ts + 1
        time.sleep(0.1)  # respect rate limit
        print(f"  Downloaded {len(all_ohlcv)} candles...", end='\r')

    df = pd.DataFrame(all_ohlcv, columns=['timestamp', 'open', 'high', 'low', 'close', 'volume'])
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms', utc=True)
    df = df.set_index('timestamp').sort_index()
    df = df[~df.index.duplicated(keep='last')]

    df.to_parquet(CACHE_FILE)
    print(f"\nDownloaded {len(df)} candles: {df.index[0]} → {df.index[-1]}")
    print(f"Cached to {CACHE_FILE}")

print(f"\nData shape: {df.shape}")
print(df.tail(3))

## Step 5 — Patch Config for T4 (reduce batch_size, enable CUDA)

In [ ]:
import yaml
import shutil
import torch
from pathlib import Path

CONFIG_PATH = Path('config/training/adversarial.yaml')
CONFIG_BACKUP = Path('config/training/adversarial.yaml.original')

assert CONFIG_PATH.exists(), f"Config not found: {CONFIG_PATH}"

# Keep a pristine backup for revert operations
if not CONFIG_BACKUP.exists():
    shutil.copy2(CONFIG_PATH, CONFIG_BACKUP)
    print(f"Original config backed up to {CONFIG_BACKUP}")

# Read current config
with open(CONFIG_PATH, 'r') as f:
    config = yaml.safe_load(f)

# Detect VRAM
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM: {vram_gb:.1f} GB → adjusting batch_size accordingly")
    # T4 has ~15 GB — batch_size 128 is fine; keep 64 for safety
    SAFE_BATCH = 64
else:
    SAFE_BATCH = 32
    print("CPU mode — using smaller batch_size=32")

# Patch for Colab compatibility
config['trader']['batch_size'] = SAFE_BATCH
config['adversary']['batch_size'] = SAFE_BATCH
# Keep n_iterations at 500 for now — search loop will override per experiment

with open(CONFIG_PATH, 'w') as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

print(f"Config patched: batch_size={SAFE_BATCH}, device={DEVICE}")
print(f"Config path: {CONFIG_PATH}")

# Global constants used in later cells
SEARCH_ITERATIONS = 50    # iterations per search experiment (fast)
CHAMPION_ITERATIONS = 500 # final champion retrain (full)
TIMEOUT_SECONDS = 900     # 15 min hard timeout per experiment
MAX_EXPERIMENTS = 20      # maximum hyperparameter combos to try
SHARPE_MIN = -5.0
SHARPE_MAX = 20.0
RESULTS_FILE = Path('/content/results_bitcoin4traders.tsv')

print(f"\nSearch settings:")
print(f"  Iterations per experiment : {SEARCH_ITERATIONS}")
print(f"  Timeout per experiment    : {TIMEOUT_SECONDS}s")
print(f"  Max experiments           : {MAX_EXPERIMENTS}")
print(f"  Results file              : {RESULTS_FILE}")

## Step 6 — Baseline Training Run (50 iterations)

In [ ]:
import subprocess
import sys
from pathlib import Path

LOG_DIR = Path('logs/training')
LOG_DIR.mkdir(parents=True, exist_ok=True)

BASELINE_LOG = Path('/content/baseline_run.log')

print(f"Starting baseline training ({SEARCH_ITERATIONS} iterations)...")
print(f"Log: {BASELINE_LOG}")
print("(This takes ~5-8 minutes on T4)\n")

cmd = [
    sys.executable, 'train.py',
    '--config', str(CONFIG_PATH),
    '--iterations', str(SEARCH_ITERATIONS),
    '--device', DEVICE,
    '--use-cached',
]

try:
    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        timeout=TIMEOUT_SECONDS,
        cwd=str(Path.cwd()),
    )
    baseline_log = result.stdout + result.stderr
    BASELINE_LOG.write_text(baseline_log)

    if result.returncode != 0:
        print("WARNING: Training process exited with non-zero code:", result.returncode)
        print("Last 50 lines of output:")
        print('\n'.join(baseline_log.splitlines()[-50:]))
    else:
        print("Baseline training finished.")
        print("Last 20 lines:")
        print('\n'.join(baseline_log.splitlines()[-20:]))

except subprocess.TimeoutExpired as e:
    baseline_log = (e.stdout or '') + (e.stderr or '')
    BASELINE_LOG.write_text(baseline_log)
    print(f"WARNING: Baseline timed out after {TIMEOUT_SECONDS}s — partial log saved.")

## Step 7 — Extract Baseline Sharpe Ratio

In [ ]:
import re

def extract_sharpe(log_text: str) -> float | None:
    """
    Parse Sharpe Ratio from training log.
    Primary pattern : 'Mean Sharpe: X.XX'  (from adversarial_trainer.py:1099)
    Fallback pattern: 'sharpe' anywhere near a float
    Returns None if nothing found or value outside sanity range.
    """
    # Primary: capture last occurrence of 'Mean Sharpe: X.XX'
    matches = re.findall(r'Mean Sharpe:\s*([\-\d\.]+)', log_text)
    if matches:
        try:
            val = float(matches[-1])
            if SHARPE_MIN <= val <= SHARPE_MAX:
                return val
            else:
                print(f"  Sharpe {val} outside sanity range [{SHARPE_MIN}, {SHARPE_MAX}] — treating as CRASH")
                return None
        except ValueError:
            pass

    # Fallback: last 'sharpe_ratio: X.XX' or 'sharpe: X.XX'
    fallback = re.findall(r'sharpe(?:_ratio)?[\s:=]+([\-\d\.]+)', log_text, re.IGNORECASE)
    if fallback:
        try:
            val = float(fallback[-1])
            if SHARPE_MIN <= val <= SHARPE_MAX:
                print(f"  (Using fallback Sharpe pattern: {val})")
                return val
        except ValueError:
            pass

    # Second fallback: use mean cumulative reward
    reward_matches = re.findall(r'Mean Reward:\s*([\-\d\.]+)', log_text)
    if reward_matches:
        try:
            val = float(reward_matches[-1])
            print(f"  No Sharpe found — using Mean Reward as proxy: {val}")
            return val  # reward has no fixed sanity range, but log it
        except ValueError:
            pass

    return None


# Read baseline log
baseline_log_text = BASELINE_LOG.read_text() if BASELINE_LOG.exists() else baseline_log
baseline_sharpe = extract_sharpe(baseline_log_text)

if baseline_sharpe is None:
    print("WARNING: Could not extract baseline Sharpe from log.")
    print("The search loop will use 0.0 as baseline (any improvement counts).")
    baseline_sharpe = 0.0
else:
    print(f"Baseline Sharpe Ratio: {baseline_sharpe:.4f}")

best_sharpe = baseline_sharpe
best_config_snapshot = None  # will be set when first improvement found
print(f"Starting search with best_sharpe = {best_sharpe:.4f}")

## Step 8 — Autonomous Hyperparameter Search Loop

**How to use:** Run the **shared helpers cell** first, then run **exactly one** of the three option cells.

| | Step 8A | Step 8B | Step 8C |
|---|---|---|---|
| **API key** | None required | Gemini (free) | Anthropic (paid) |
| **Intelligence** | Grid search | LLM-guided | LLM-guided (best) |
| **Model** | — | `gemini-2.0-flash` | `claude-sonnet-4-6` |
| **Colab Secret needed** | — | `GEMINI_API_KEY` | `ANTHROPIC_API_KEY` |

Each experiment:
1. Picks next combo (grid) **or** asks LLM to suggest a YAML change
2. Patches `config/training/adversarial.yaml`
3. Runs `train.py --iterations 50`
4. Extracts Sharpe → keeps if improved, reverts if worse
5. Logs everything to `results_bitcoin4traders.tsv`

In [ ]:
# ── Step 8: Shared helpers (run this first, then run ONE of the option cells below) ──

import yaml
import shutil
import itertools
import random
import subprocess
import sys
import re
import csv
import matplotlib.pyplot as plt
from datetime import datetime, timezone
from pathlib import Path
from copy import deepcopy
from IPython.display import clear_output, display

# ── Allowed search space ───────────────────────────────────────────────────────
ALLOWED_PATCHES = {
    'trader': {
        'actor_lr':     (float, (1e-6, 1e-2)),
        'entropy_coef': (float, (0.001, 0.5)),
        'hidden_dim':   (int,   [64, 128, 256, 512]),
        'clip_epsilon': (float, (0.05, 0.5)),
        'n_epochs':     (int,   (1, 30)),
        'batch_size':   (int,   [32, 64, 128]),
        'target_kl':    (float, (0.005, 0.05)),
    },
    'adversary': {
        'actor_lr':     (float, (1e-6, 1e-2)),
        'entropy_coef': (float, (0.001, 0.5)),
    },
    'training': {
        'adversary_strength': (float, (0.0, 1.0)),
    }
}

# ── Config helpers ─────────────────────────────────────────────────────────────

def read_config() -> dict:
    with open(CONFIG_PATH, 'r') as f:
        return yaml.safe_load(f)

def write_config(cfg: dict):
    with open(CONFIG_PATH, 'w') as f:
        yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)

def run_training(n_iterations: int, log_path: Path) -> tuple:
    cmd = [
        sys.executable, 'train.py',
        '--config', str(CONFIG_PATH),
        '--iterations', str(n_iterations),
        '--device', DEVICE,
        '--use-cached',
    ]
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=TIMEOUT_SECONDS)
        log = result.stdout + result.stderr
    except subprocess.TimeoutExpired as e:
        log = (e.stdout or '') + (e.stderr or '')
        log_path.write_text(log)
        print(f"  TIMEOUT after {TIMEOUT_SECONDS}s")
        return None, log
    except Exception as ex:
        log = str(ex)
        log_path.write_text(log)
        print(f"  EXCEPTION: {ex}")
        return None, log
    log_path.write_text(log)
    return extract_sharpe(log), log

def log_result(row: dict):
    write_header = not RESULTS_FILE.exists()
    with open(RESULTS_FILE, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()), delimiter='\t')
        if write_header:
            writer.writeheader()
        writer.writerow(row)

def apply_and_evaluate(section, key, value, exp_num, label, best_sharpe, best_snapshot):
    cfg = read_config()
    if section not in cfg or key not in cfg.get(section, {}):
        print(f"  SKIP: {section}.{key} not found in YAML")
        return best_sharpe, best_snapshot, 'SKIP'
    cfg[section][key] = value
    write_config(cfg)

    exp_log = Path(f'/content/exp_{exp_num:03d}.log')
    sharpe, _ = run_training(SEARCH_ITERATIONS, exp_log)

    old_best = best_sharpe
    if sharpe is None:
        outcome = 'CRASH'
        print(f"  → CRASH — reverting")
        write_config(best_snapshot) if best_snapshot else shutil.copy2(CONFIG_BACKUP, CONFIG_PATH)
    elif sharpe > best_sharpe:
        outcome = 'IMPROVED'
        print(f"  → IMPROVED {best_sharpe:.4f} → {sharpe:.4f} (+{sharpe - best_sharpe:.4f})")
        best_sharpe = sharpe
        best_snapshot = deepcopy(cfg)
    else:
        outcome = 'WORSE'
        print(f"  → WORSE ({sharpe:.4f}, Δ={sharpe - best_sharpe:.4f}) — reverting")
        write_config(best_snapshot) if best_snapshot else shutil.copy2(CONFIG_BACKUP, CONFIG_PATH)

    log_result({
        'experiment':  exp_num,
        'timestamp':   datetime.now(timezone.utc).replace(tzinfo=None).isoformat(),
        'patch':       label,
        'sharpe':      round(sharpe, 4) if sharpe is not None else 'CRASH',
        'delta':       round(sharpe - old_best, 4) if sharpe is not None else float('nan'),
        'outcome':     outcome,
        'best_so_far': round(best_sharpe, 4),
    })
    return best_sharpe, best_snapshot, outcome

def plot_progress(results_file: Path):
    if not results_file.exists():
        return
    import pandas as pd
    df = pd.read_csv(results_file, sep='\t')
    df_plot = df[df['outcome'] != 'BASELINE']
    if df_plot.empty:
        return
    fig, ax = plt.subplots(figsize=(10, 3))
    sharpes = pd.to_numeric(df_plot['sharpe'], errors='coerce')
    bests   = pd.to_numeric(df_plot['best_so_far'], errors='coerce')
    ax.scatter(df_plot['experiment'], sharpes, c='steelblue', s=40, label='Experiment Sharpe')
    ax.plot(df_plot['experiment'], bests, 'g-', linewidth=2, label='Best so far')
    ax.axhline(y=float(df[df['outcome']=='BASELINE']['sharpe'].iloc[0]),
               color='orange', linestyle='--', label='Baseline')
    ax.set_xlabel('Experiment'); ax.set_ylabel('Sharpe Ratio')
    ax.set_title('Hyperparameter Search Progress')
    ax.legend(); plt.tight_layout()
    clear_output(wait=True)
    display(fig)
    plt.close(fig)

# ── LLM helpers (used by Options B + C) ───────────────────────────────────────

def validate_llm_suggestion(section, key, raw_value):
    if section not in ALLOWED_PATCHES:
        raise ValueError(f"Section '{section}' not in allowed list")
    if key not in ALLOWED_PATCHES[section]:
        raise ValueError(f"Key '{key}' not allowed in section '{section}'")
    typ, constraint = ALLOWED_PATCHES[section][key]
    try:
        value = typ(raw_value)
    except (ValueError, TypeError):
        raise ValueError(f"Cannot cast '{raw_value}' to {typ.__name__}")
    if isinstance(constraint, list):
        if value not in constraint:
            raise ValueError(f"{value} not in allowed list {constraint}")
    else:
        lo, hi = constraint
        if not (lo <= value <= hi):
            raise ValueError(f"{value} outside range [{lo}, {hi}]")
    return section, key, value

def parse_llm_reply(reply):
    section = key = raw_value = None
    for line in reply.splitlines():
        line = line.strip()
        if line.upper().startswith('SECTION:'):
            section = line.split(':', 1)[1].strip().lower()
        elif line.upper().startswith('KEY:'):
            key = line.split(':', 1)[1].strip().lower()
        elif line.upper().startswith('VALUE:'):
            raw_value = line.split(':', 1)[1].strip()
    if not all([section, key, raw_value]):
        raise ValueError(f"Could not parse SECTION/KEY/VALUE from:\n{reply}")
    return section, key, raw_value

LLM_PROMPT_TEMPLATE = """You are an ML research agent optimizing a RL trading system.
Goal: maximize Sharpe Ratio (higher = better risk-adjusted returns).

Current adversarial.yaml config:
{yaml_content}

Experiment history:
{history}

Last training log (last 80 lines):
{log_tail}

Allowed changes (section → key → type):
trader: actor_lr (float 1e-6..1e-2), entropy_coef (float 0.001..0.5),
        hidden_dim (int 64/128/256/512), clip_epsilon (float 0.05..0.5),
        n_epochs (int 1..30), batch_size (int 32/64/128), target_kl (float 0.005..0.05)
adversary: actor_lr (float 1e-6..1e-2), entropy_coef (float 0.001..0.5)
training: adversary_strength (float 0.0..1.0)

Suggest ONE change to improve the Sharpe Ratio.
Reply ONLY in this exact format (no explanation):
SECTION: <section_name>
KEY: <key_name>
VALUE: <new_value>"""

# Write BASELINE row exactly once
if not RESULTS_FILE.exists():
    log_result({
        'experiment': 0,
        'timestamp': datetime.now(timezone.utc).replace(tzinfo=None).isoformat(),
        'patch': 'BASELINE', 'sharpe': round(best_sharpe, 4),
        'delta': 0.0, 'outcome': 'BASELINE', 'best_so_far': round(best_sharpe, 4),
    })
    print(f"Shared helpers loaded. Baseline Sharpe = {best_sharpe:.4f} (BASELINE logged)")
else:
    print(f"Shared helpers loaded. Baseline Sharpe = {best_sharpe:.4f} (BASELINE already exists)")

### Step 8A — Option A: Grid Search (no API key required)

Run **only this cell** if you have no API key. Uses a fixed grid of hyperparameter values, one param at a time.

In [ ]:
# ── Option A: Grid Search (no API key) ────────────────────────────────────────
# Run Steps 1–8 (shared helpers), then run THIS cell only.

SEARCH_PARAMS = {
    'actor_lr':     [5e-5, 1e-4, 2e-4, 3e-4],
    'entropy_coef': [0.05, 0.08, 0.12, 0.15],
    'hidden_dim':   [64, 128, 256],
    'clip_epsilon': [0.1, 0.15, 0.2, 0.3],
}
candidates = [
    ('trader', param, v)
    for param, values in SEARCH_PARAMS.items()
    for v in values
]
random.seed(42)
random.shuffle(candidates)
candidate_cycle = itertools.cycle(candidates)

print(f"[Option A] {len(candidates)} single-param candidates (cycled), max={MAX_EXPERIMENTS}\n")

print(f"{'='*70}")
print(f"OPTION A — GRID SEARCH | baseline Sharpe = {best_sharpe:.4f}")
print(f"{'='*70}\n")

exp_num = 0
for section, key, value in candidate_cycle:
    if exp_num >= MAX_EXPERIMENTS:
        print(f"Reached MAX_EXPERIMENTS={MAX_EXPERIMENTS} — stopping.")
        break
    exp_num += 1
    label = f"{section}.{key}={value}"
    print(f"\n[Exp {exp_num}/{MAX_EXPERIMENTS}] {label}")
    best_sharpe, best_config_snapshot, _ = apply_and_evaluate(
        section, key, value, exp_num, label, best_sharpe, best_config_snapshot
    )
    plot_progress(RESULTS_FILE)

print(f"\n{'='*70}")
print(f"OPTION A COMPLETE | Best Sharpe: {best_sharpe:.4f}")
print(f"{'='*70}")

### Step 8B — Option B: Gemini Free Tier (LLM-guided)

Run **only this cell** if you have a Gemini API key.
Add `GEMINI_API_KEY` to **Colab Secrets** (🔑 icon in the left sidebar) before running.

In [ ]:
# ── Option B: Gemini Free Tier ─────────────────────────────────────────────────
# Run Steps 1–8 (shared helpers), then run THIS cell only.
# Requires GEMINI_API_KEY in Colab Secrets.

get_ipython().system('pip install google-generativeai -q')
import google.generativeai as genai
from google.colab import userdata
genai.configure(api_key=userdata.get('GEMINI_API_KEY'))
gemini_model = genai.GenerativeModel('gemini-2.0-flash')
print("Gemini model ready.")

history_rows = []
last_log_text = BASELINE_LOG.read_text() if BASELINE_LOG.exists() else ''

print(f"\n{'='*70}")
print(f"OPTION B — GEMINI-GUIDED SEARCH | baseline Sharpe = {best_sharpe:.4f}")
print(f"{'='*70}\n")

for exp_num in range(1, MAX_EXPERIMENTS + 1):
    yaml_content = Path(CONFIG_PATH).read_text()
    history_str  = '\n'.join(history_rows[-10:]) or '(none yet)'
    log_tail_str = '\n'.join(last_log_text.splitlines()[-80:])

    prompt = LLM_PROMPT_TEMPLATE.format(
        yaml_content=yaml_content, history=history_str, log_tail=log_tail_str
    )

    print(f"\n[Exp {exp_num}/{MAX_EXPERIMENTS}] Asking Gemini...")
    try:
        response = gemini_model.generate_content(prompt)
        reply    = response.text.strip()
        print(f"  Reply: {reply[:120]}")
        section, key, raw_value = parse_llm_reply(reply)
        section, key, value     = validate_llm_suggestion(section, key, raw_value)
        label = f"{section}.{key}={value}"
        print(f"  Suggestion: {label}")
    except Exception as ex:
        print(f"  LLM/parse error: {ex} — skipping")
        history_rows.append(f"Exp {exp_num}: SKIP ({ex})")
        continue

    best_sharpe, best_config_snapshot, outcome = apply_and_evaluate(
        section, key, value, exp_num, label, best_sharpe, best_config_snapshot
    )
    history_rows.append(f"Exp {exp_num}: {label} → {outcome} (best={best_sharpe:.4f})")
    exp_log = Path(f'/content/exp_{exp_num:03d}.log')
    if exp_log.exists():
        last_log_text = exp_log.read_text()
    plot_progress(RESULTS_FILE)

print(f"\n{'='*70}")
print(f"OPTION B COMPLETE | Best Sharpe: {best_sharpe:.4f}")
print(f"{'='*70}")

### Step 8C — Option C: Claude / Anthropic API (most intelligent)

Run **only this cell** if you have an Anthropic API key.
Add `ANTHROPIC_API_KEY` to **Colab Secrets** (🔑 icon in the left sidebar) before running.

In [ ]:
# ── Option C: Claude / Anthropic API ──────────────────────────────────────────
# Run Steps 1–8 (shared helpers), then run THIS cell only.
# Requires ANTHROPIC_API_KEY in Colab Secrets.

get_ipython().system('pip install anthropic -q')
import anthropic
from google.colab import userdata
client = anthropic.Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))
print("Anthropic client ready.")

history_rows = []
last_log_text = BASELINE_LOG.read_text() if BASELINE_LOG.exists() else ''

print(f"\n{'='*70}")
print(f"OPTION C — CLAUDE-GUIDED SEARCH | baseline Sharpe = {best_sharpe:.4f}")
print(f"{'='*70}\n")

for exp_num in range(1, MAX_EXPERIMENTS + 1):
    yaml_content = Path(CONFIG_PATH).read_text()
    history_str  = '\n'.join(history_rows[-10:]) or '(none yet)'
    log_tail_str = '\n'.join(last_log_text.splitlines()[-80:])

    prompt = LLM_PROMPT_TEMPLATE.format(
        yaml_content=yaml_content, history=history_str, log_tail=log_tail_str
    )

    print(f"\n[Exp {exp_num}/{MAX_EXPERIMENTS}] Asking Claude...")
    try:
        message = client.messages.create(
            model='claude-sonnet-4-6',
            max_tokens=256,
            messages=[{'role': 'user', 'content': prompt}],
        )
        reply    = message.content[0].text.strip()
        print(f"  Reply: {reply[:120]}")
        section, key, raw_value = parse_llm_reply(reply)
        section, key, value     = validate_llm_suggestion(section, key, raw_value)
        label = f"{section}.{key}={value}"
        print(f"  Suggestion: {label}")
    except Exception as ex:
        print(f"  LLM/parse error: {ex} — skipping")
        history_rows.append(f"Exp {exp_num}: SKIP ({ex})")
        continue

    best_sharpe, best_config_snapshot, outcome = apply_and_evaluate(
        section, key, value, exp_num, label, best_sharpe, best_config_snapshot
    )
    history_rows.append(f"Exp {exp_num}: {label} → {outcome} (best={best_sharpe:.4f})")
    exp_log = Path(f'/content/exp_{exp_num:03d}.log')
    if exp_log.exists():
        last_log_text = exp_log.read_text()
    plot_progress(RESULTS_FILE)

print(f"\n{'='*70}")
print(f"OPTION C COMPLETE | Best Sharpe: {best_sharpe:.4f}")
print(f"{'='*70}")

## Step 9 — Champion Retrain (Full 500 Iterations)

In [ ]:
CHAMPION_LOG = Path('/content/champion_run.log')
CHAMPION_TIMEOUT = 7200  # 2 hours for full retrain

# Ensure champion config is written
if best_config_snapshot is not None:
    write_config(best_config_snapshot)
    print(f"Champion config applied. Starting full retrain ({CHAMPION_ITERATIONS} iterations)...")
else:
    # Restore original config
    shutil.copy2(CONFIG_BACKUP, CONFIG_PATH)
    print(f"No improvement found. Retraining with original config ({CHAMPION_ITERATIONS} iterations)...")

# Patch n_iterations in config
champ_cfg = read_config()
# (n_iterations is overridden via --iterations CLI arg, config value is backup)

print(f"Log: {CHAMPION_LOG}")
print("(Full retrain takes ~45-90 min on T4)\n")

cmd = [
    sys.executable, 'train.py',
    '--config', str(CONFIG_PATH),
    '--iterations', str(CHAMPION_ITERATIONS),
    '--device', DEVICE,
    '--use-cached',
]

try:
    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        timeout=CHAMPION_TIMEOUT,
    )
    champ_log = result.stdout + result.stderr
    CHAMPION_LOG.write_text(champ_log)

    final_sharpe = extract_sharpe(champ_log)
    if final_sharpe is not None:
        print(f"\nChampion retrain complete!")
        print(f"Final Sharpe: {final_sharpe:.4f}")
        print(f"(vs. search baseline: {baseline_sharpe:.4f}, Δ={final_sharpe - baseline_sharpe:.4f})")
    else:
        print("Could not extract final Sharpe from champion log.")
        print("Last 30 lines:")
        print('\n'.join(champ_log.splitlines()[-30:]))

except subprocess.TimeoutExpired as e:
    champ_log = (e.stdout or '') + (e.stderr or '')
    CHAMPION_LOG.write_text(champ_log)
    print(f"Champion retrain timed out after {CHAMPION_TIMEOUT}s — partial model saved.")

## Step 10 — View Results Table

In [ ]:
import pandas as pd

if RESULTS_FILE.exists():
    results_df = pd.read_csv(RESULTS_FILE, sep='\t')
    print(f"Results: {len(results_df)} rows")

    # Style the table
    def highlight_outcome(row):
        color = ''
        if row['outcome'] == 'IMPROVED':
            color = 'background-color: #c6efce'  # green
        elif row['outcome'] == 'CRASH':
            color = 'background-color: #ffc7ce'  # red
        elif row['outcome'] == 'WORSE':
            color = 'background-color: #ffeb9c'  # yellow
        return [color] * len(row)

    display(results_df.style.apply(highlight_outcome, axis=1))

    # Summary
    print(f"\nSummary:")
    print(results_df['outcome'].value_counts().to_string())
    improved = results_df[results_df['outcome'] == 'IMPROVED']
    if len(improved) > 0:
        print(f"\nBest experiment:")
        best_row = improved.loc[improved['sharpe'].astype(float).idxmax()]
        print(best_row.to_string())
else:
    print(f"Results file not found: {RESULTS_FILE}")

## Step 11 — Save Results & Best Model to Google Drive

In [ ]:
import shutil
from datetime import datetime
from pathlib import Path

DRIVE_OUTPUT_DIR = Path('/content/drive/MyDrive/BITCOIN4Traders_results')
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

timestamp = datetime.utcnow().strftime('%Y%m%d_%H%M%S')

saved = []

# 1. Results TSV
if RESULTS_FILE.exists():
    dest = DRIVE_OUTPUT_DIR / f'results_{timestamp}.tsv'
    shutil.copy2(RESULTS_FILE, dest)
    saved.append(str(dest))

# 2. Champion config
if CONFIG_PATH.exists():
    dest = DRIVE_OUTPUT_DIR / f'adversarial_champion_{timestamp}.yaml'
    shutil.copy2(CONFIG_PATH, dest)
    saved.append(str(dest))

# 3. Champion training log
if CHAMPION_LOG.exists():
    dest = DRIVE_OUTPUT_DIR / f'champion_run_{timestamp}.log'
    shutil.copy2(CHAMPION_LOG, dest)
    saved.append(str(dest))

# 4. Best model checkpoint (largest .pth in checkpoint dir)
checkpoint_dir = Path('data/models/adversarial')
if checkpoint_dir.exists():
    pth_files = sorted(checkpoint_dir.glob('*.pth'), key=lambda p: p.stat().st_mtime)
    if pth_files:
        latest_ckpt = pth_files[-1]
        dest = DRIVE_OUTPUT_DIR / f'champion_model_{timestamp}_{latest_ckpt.name}'
        shutil.copy2(latest_ckpt, dest)
        saved.append(str(dest))
        print(f"Saved checkpoint: {latest_ckpt.name} ({latest_ckpt.stat().st_size / 1e6:.1f} MB)")
    else:
        print(f"No .pth checkpoints found in {checkpoint_dir}")
else:
    print(f"Checkpoint directory not found: {checkpoint_dir}")

print(f"\nSaved {len(saved)} file(s) to Drive:")
for f in saved:
    print(f"  {f}")

print("\nDone! Session complete.")